In [1]:
from torchvision import models
from torchvision.models import vit_b_16

print("=" * 60)

try:
    m1 = models.resnet50(weights=None)
    print("ResNet50 build success")
except Exception as e:
    print("Fail", e)

try:
    m2 = vit_b_16(weights=None)
    print("Vit-B/16 build success")
except Exception as e:
    print("Fail",e)

print("=" * 60)

ResNet50 build success
Vit-B/16 build success


In [2]:
# 1) 환경 설정 + import
import os
import gc
import random
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, f1_score, roc_auc_score, recall_score
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import vit_b_16, ViT_B_16_Weights, ResNet50_Weights

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

torch.backends.cudnn.benchmark = True

Device: cuda


In [3]:
print(f"현재 작업 경로: {os.getcwd()}")

test_relative = '/tf/nasw/dataset001/preprocessed/npz/fold_1_train.npz'
print(f"상대경로: {os.path.exists(test_relative)}")

현재 작업 경로: /tf/notebooks/Image Encoder
상대경로: False


In [4]:
#2. 경로 설정
DATA_ROOT = Path("/tf/nasw/dataset001/preprocessed/npz_us")

FOLDS = [1, 2, 3, 4, 5]

OUTPUT_DIR = Path("./ct_multiclass_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 8
NUM_WORKERS = 2
NUM_EPOCHS = 30
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 7

NUM_CLASSES = 10

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("FOLDS:", FOLDS)

DATA_ROOT: /tf/nasw/dataset001/preprocessed/npz_us
OUTPUT_DIR: /tf/notebooks/Image Encoder/ct_multiclass_models
FOLDS: [1, 2, 3, 4, 5]


In [5]:
def inspect_fold_npz(fold_idx):
    train_npz_path = DATA_ROOT / f"fold_{fold_idx}_train.npz"
    val_npz_path = DATA_ROOT / f"fold_{fold_idx}_val.npz"

    print("TRAIN:", train_npz_path, train_npz_path.exits())
    print("VAL:", val_npz_path, val_npz_path.exists())

    train_npz = np.load(train_npz_path, allow_pickle=False, mmap_mode="r")
    val_npz = np.load(val_npz_path, allow_pickle=False, mmap_mode="r")

    for k in train_npz.files:
        print(f"[train] {k}: shape={train_npz[k].shape}, dtype={train_npz[k].dtype}")
    for k in val_npz.files:
        print(f"[val] {k}: shape={val_npz[k].shape}, dtype={val_npz[k].dtype}")

In [6]:
#로컬 pretrained weight 경로
LOCAL_RESNET_WEIGHTS = Path("/tf/models/pretrained/resnet50-11ad3fa6.pth")
LOCAL_VIT_WEIGHTS = Path("/tf/models/pretrained/vit_b_16-c867db91.pth")

print("LOCAL_RESNET_WEIGHTS exists:", LOCAL_RESNET_WEIGHTS.exists())
print("LOCAL_VIT_WEIGHTS exists:", LOCAL_VIT_WEIGHTS.exists())

LOCAL_RESNET_WEIGHTS exists: False
LOCAL_VIT_WEIGHTS exists: False


In [7]:
def load_local_state_dict(weight_path):
    weight_path = Path(weight_path)
    assert weight_path.exists(), f"Weight file not found: {weight_path}"

    ckpt = torch.load(weight_path, map_location="cpu")

    if isinstance(ckpt, dict) and "state_dict" in ckpt and isinstance(ckpt["state_dict"], dict):
        ckpt = ckpt["state_dict"]

    return ckpt

In [8]:
#4. Dataset 정의
class OvarianCTNPZDataset(Dataset):
    def __init__(self, npz_path):
        super().__init__()
        self.npz = np.load(npz_path, allow_pickle=False, mmap_mode="r")

        self.images = self.npz["images"]
        self.labels = self.npz["labels"].astype(np.float32)

        self.mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1)

        self.tumor_types = self.npz["tumor_types"] if "tumor_types" in self.npz.files else None
        self.recurrences = self.npz["recurrences"] if "recurrences" in self.npz.files else None

    def __len__(self):
        return len(self.tumor_types)

    def __getitem__(self, idx):
        x = self.images[idx]
        y = self.tumor_types[idx]

        x = torch.from_numpy(x).float().permute(0, 3, 1, 2).contiguous()

        x = x / 255.0
        x = (x-self.mean) / self.std

        y = torch.tensor(int(y), dtype=torch.long)

        return x, y

In [9]:
# fold별 dataLoader 생성
def make_fold_loaders(fold_idx):
    train_npz_path = DATA_ROOT / f"fold_{fold_idx}_train.npz"
    val_npz_path = DATA_ROOT / f"fold_{fold_idx}_val.npz"

    assert train_npz_path.exists(), f"Missing: {train_npz_path}"
    assert val_npz_path.exists(), f"Missing: {val_npz_path}"

    train_dataset = OvarianCTNPZDataset(train_npz_path)
    val_dataset = OvarianCTNPZDataset(val_npz_path)

    train_loader = DataLoader(
        train_dataset,
        batch_size = BATCH_SIZE,
        shuffle = True,
        num_workers = NUM_WORKERS,
        pin_memory = True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size = BATCH_SIZE,
        shuffle = False,
        num_workers = NUM_WORKERS,
        pin_memory = True
    )

    tumor_np = train_dataset.tumor_types.astype(int)
    num_classes = len(np.unique(tumor_np))
    class_counts = np.bincount(tumor_np, minlength=num_classes)
    class_weights = 1.0 / np.maximum(class_counts, 1)
    class_weights = class_weights / class_weights.sum() * num_classes
    class_weights = torch.tensor(class_weights, device=device, dtype=torch.float32)
    
    print(f"[FOLD {fold_idx}] Tumor type counts: {class_counts}, weights: {class_weights}")

    return train_loader, val_loader, class_weights

In [10]:
#6 train/eval 함수

scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))

def compute_binary_metrics(y_true, y_prob, threshold = 0.5):
    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob).astype(float)
    y_pred = (y_prob > threshold).astype(int)

    metrics = {
        "acc": accuracy_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.5,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "cm": confusion_matrix(y_true, y_pred)
    }
    return metrics

def compute_multiclass_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    if len(y_true) == 0:
        return {
            "acc": np.nan,
            "macro_f1": np.nan,
            "cm": None
        }
    return {
        "acc": accuracy_score(y_true, y_pred),
         "cm": confusion_matrix(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division = 0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0)
    }


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    y_true, y_pred = [], []

    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            logits = model(imgs)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)

        preds = torch.argmax(logits, dim=1)
        y_true.extend(labels.detach().cpu().numpy().ravel())
        y_pred.extend(preds.detach().cpu().numpy().ravel())

    metric_vals = compute_multiclass_metrics(y_true, y_pred)

    metrics = {
        "loss": running_loss / len(loader.dataset),
        "acc": metric_vals["acc"],
        "macro_f1": metric_vals["macro_f1"],
        "macro_recall": metric_vals["macro_recall"],
        "cm": metric_vals["cm"]
    }
    return metrics

@torch.no_grad()
def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    y_true, y_pred = [], []

    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking = True)
        labels = labels.to(device, non_blocking = True)

        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            logits = model(imgs)
            loss = criterion(logits, labels)

        running_loss += loss.item() * imgs.size(0)

        preds = torch.argmax(logits, dim=1)
        y_true.extend(labels.detach().cpu().numpy().ravel())
        y_pred.extend(preds.detach().cpu().numpy().ravel())

    metric_vals = compute_multiclass_metrics(y_true, y_pred)

    metrics = {
        "loss": running_loss / len(loader.dataset),
        "acc": metric_vals["acc"],
        "macro_f1": metric_vals["macro_f1"],
        "macro_recall": metric_vals["macro_recall"],
        "cm": metric_vals["cm"]
    }
    return metrics

@torch.no_grad()
def eval_confusion(model, loader, device, threshold = 0.5):
    model.eval()
    y_true, y_pred = [], []

    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking = True)
        labels = labels.to(device, non_blocking = True)

        logits = model(imgs)
        preds = torch.argmax(logits, dim=1)
        y_true.extend(labels.cpu().numpy().ravel())
        y_pred.extend(preds.cpu().numpy().ravel())

    metric_vals = compute_multiclass_metrics(y_true, y_pred)

    print(f"\m--Final Evaluation--")
    print(f"Accuracy: {metric_vals['acc']:.4f}")
    print(f"macro F1-score : {metric_vals['macro_f1']:.4f}")
    print(f"macro Recall : {metric_vals['macro_recall']:.4f}")
    print(metric_vals["cm"])

    print("\nClassification report:")
    print(classification_report(np.array(y_true).astype(int), y_pred, digits = 4, zero_division = 0))

    return metric_vals
        

In [11]:
# slice encoder + patient-level pooling wrapper

class PatientSliceAttentionClassifier(nn.Module):
    def __init__(self, encoder, feat_dim, num_classes, hidden_dim=512, dropout=0.3):
        super().__init__()
        self.encoder = encoder

        self.attn = nn.Sequential(
            nn.Linear(feat_dim, feat_dim // 2),
            nn.Tanh(),
            nn.Linear(feat_dim // 2, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        # x: (B, S, C, H, W)
        B, S, C, H, W = x.shape
        x = x.view(B * S, C, H, W)

        feat = self.encoder(x)
        feat = feat.view(B, S, -1)

        attn_score = self.attn(feat)
        attn_weight = torch.softmax(attn_score, dim=1)

        pooled = (feat * attn_weight).sum(dim=1)
        logits = self.classifier(pooled)

        return logits

In [12]:
#7 모델 정의
def build_resnet50_patient_model(num_classes, load_pretrained=True, freeze_backbone=True, unfreeze_layer4=True, weight_path=LOCAL_RESNET_WEIGHTS):
    encoder = models.resnet50(weights=None)

    if load_pretrained:
        state_dict = load_local_state_dict(weight_path)
        encoder.load_state_dict(state_dict, strict = True)
        
    feat_dim = encoder.fc.in_features
    encoder.fc = nn.Identity()

    if freeze_backbone:
        for p in encoder.parameters():
            p.requires_grad = False

        if unfreeze_layer4:
            for p in encoder.layer4.parameters():
                p.requires_grad = True
    
    model = PatientSliceAttentionClassifier(
        encoder = encoder,
        num_classes = num_classes,
        feat_dim = feat_dim,
        hidden_dim = 512,
        dropout=0.1
    )
    
    return model

def build_vit_patient_model(num_classes, load_pretrained=True, weight_path=LOCAL_VIT_WEIGHTS):
    encoder = vit_b_16(weights=None)
    if load_pretrained:
        state_dict = load_local_state_dict(weight_path)
        encoder.load_state_dict(state_dict, strict = True)
    
    feat_dim = encoder.heads.head.in_features
    encoder.heads = nn.Identity()
    
    model = PatientSliceAttentionClassifier(
        encoder = encoder,
        feat_dim = feat_dim,
        num_classes = num_classes,
        hidden_dim = 512,
        dropout=0.3
    )
    return model

class CNNTransformerHybridPatient(nn.Module):
    def __init__(self, num_classes, d_model=512, nhead=8, num_layers=2, dim_feedforward=1024, dropout=0.3, load_pretrained=True, freeze_backbone=True, unfreeze_layer4=True, resnet_weight_path=LOCAL_RESNET_WEIGHTS):
        super().__init__()
        backbone = models.resnet50(weights=None)
        if load_pretrained:
            state_dict = load_local_state_dict(resnet_weight_path)
            backbone.load_state_dict(state_dict, strict = True)
        feat_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.encoder = backbone

        if freeze_backbone:
            for p in self.encoder.parameters():
                p.requires_grad = False

            if unfreeze_layer4:
                for p in self.encoder.layer4.parameters():
                    p.requires_grad = True

        self.proj = nn.Linear(feat_dim, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1+8, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=0.1,
            batch_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers= num_layers)
        self.norm = nn.LayerNorm(d_model)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        B, S, C, H, W = x.shape
        x = x.view(B * S, C, H, W)

        feat = self.encoder(x)
        feat = feat.view(B, S, -1)
        feat = self.proj(feat)

        cls = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, feat], dim=1)
        tokens = tokens + self.pos_embed[:, :tokens.size(1), :]

        tokens = self.transformer(tokens)
        tokens = self.norm(tokens)

        cls_out = tokens[:, 0, :]
        logits = self.classifier(cls_out)

        return logits

        

def build_hybrid_patient_model(load_pretrained=True):
    return CNNTransformerHybridPatient(
        num_classes = num_classes,
        load_pretrained=load_pretrained,
        freeze_backbone = True,
        unfreeze_layer4 = True,
        resnet_weight_path=LOCAL_RESNET_WEIGHTS
    )

In [13]:
#8 공통 학습 함수
def fit_model(model, model_name, fold_idx, train_loader, val_loader, device, num_epochs=30, lr = 1e-4, weight_decay=1e-4, class_weights=None, patience = 7, min_delta=0.001):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr = lr,
        weight_decay = weight_decay
    )

    best_macro_f1 = -1
    best_val_loss = np.inf
    best_path = OUTPUT_DIR / f"{model_name}_fold{fold_idx}_best.pth"
    history = []
    early_stop_counter = 0

    for epoch in range(1, num_epochs+1):
        train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = eval_one_epoch(model, val_loader, criterion, device)

        row = {
            "fold": fold_idx,
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["acc"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_macro_recall": train_metrics["macro_recall"],
            "val_loss":val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_macro_recall": val_metrics["macro_recall"],
        }
        history.append(row)

        print(
            f"[{model_name}][Fold {fold_idx}][Epoch {epoch:02d}]"
            f"train loss = {row['train_loss']:.4f}, acc={row['train_acc']:.4f}, "
            f"macro_f1 = {row['train_macro_f1']:.4f}, macro_recall={row['train_macro_recall']:.4f} | "
            f"val loss = {row['val_loss']:.4f}, acc = {row['val_acc']:.4f}, "
            f"macro_f1 = {row['val_macro_f1']:.4f}, macro_recall={row['val_macro_recall']:.4f}"
        )

        if row["val_macro_f1"] > best_macro_f1:
            best_macro_f1 = row["val_macro_f1"]

        if row["val_loss"] < best_val_loss - min_delta:
            best_val_loss = row["val_loss"]
            early_stop_counter = 0
            torch.save(model.state_dict(), best_path)
            print(f"  -> best saved by val_loss: {best_path}")
        else:
            early_stop_counter += 1
            print(f"  -> no improvement ({early_stop_counter}/{patience})")

        if early_stop_counter >= patience:
            print(f"  -> early stopping triggered at epoch {epoch}")
            break

    history_df = pd.DataFrame(history)
    return model, history_df, best_path

In [14]:
# model 로드 함수
def load_model(model_type, model_path, device):
    if model_type == "resnet":
        model = build_resnet50_patient_model(NUM_CLASSES)
    elif model_type == "vit":
        model = build_vit_patient_model(NUM_CLASSES)
    elif model_type == "hybrid":
        model = build_hybrid_patient_model(NUM_CLASSES)
    else:
        raise ValueError("Unknown model type")

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    return model

In [15]:
def run_cv_for_model(model_name, model_type, build_fn):
    fold_results = []
    fold_histories = []

    for fold_idx in FOLDS:
        print("\n" + "=" * 80)
        print(f"Running {model_name} | Fold {fold_idx}")
        print("=" * 80)

        train_loader, val_loader, class_weights = make_fold_loaders(fold_idx)

        model = build_fn()

        _, history_df, best_path = fit_model(
            model=model,
            model_name=model_name,
            fold_idx = fold_idx,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            num_epochs=NUM_EPOCHS,
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            class_weights=class_weights,
            patience=PATIENCE
            
        )

        best_model = load_model(model_type, best_path, device)
        final_metrics = eval_confusion(best_model, val_loader, device)

        print(f"\n[{model_name}][Fold {fold_idx}] Final Metrics")
        print(f"Accuracy: {final_metrics['acc']:.4f}")
        print(f"macro F1 : {final_metrics['macro_f1']:.4f}")
        print(f"macro Recall : {final_metrics['macro_recall']:.4f}")
        print("Confusion Matrix:")
        print(final_metrics["cm"])

        fold_results.append({
            "model": model_name,
            "fold": fold_idx,
            "acc": final_metrics["acc"],
            "macro F1-score": final_metrics["macro_f1"],
            "macro Recall": final_metrics["macro_recall"],
            "Confusion Matrix": final_metrics["cm"]
        })

        history_df["model"] = model_name
        fold_histories.append(history_df)

        del model
        del best_model
        del train_loader
        del val_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    result_df = pd.DataFrame(fold_results)
    history_df_all = pd.concat(fold_histories, ignore_index=True)
    return result_df, history_df_all

In [16]:
#9 ResNet50
print(">>> Training ResNet50...")
resnet_cv_df, resnet_history_df = run_cv_for_model(
    model_name="ResNet50",
    model_type="resnet",
    build_fn=lambda: build_resnet50_patient_model(NUM_CLASSES, load_pretrained=True)
)

display(resnet_cv_df)

>>> Training ResNet50...

Running ResNet50 | Fold 1
[FOLD 1] Tumor type counts: [ 32  22 454  67 581  25 318 249 233 491], weights: tensor([2.0967, 3.0498, 0.1478, 1.0014, 0.1155, 2.6838, 0.2110, 0.2695, 0.2880,
        0.1366], device='cuda:0')


AssertionError: Weight file not found: /tf/models/pretrained/resnet50-11ad3fa6.pth

In [ ]:
#10 vit
print(">>> Training VIT...")
vit_cv_df, vit_history_df = run_cv_for_model(
    model_name="ViT-B16",
    model_type="vit",
    build_fn=lambda: build_vit_patient_model(NUM_CLASSES, load_pretrained=True)
)

display(vit_cv_df)

In [ ]:
#11 Hybrid
print(">>> Training Hybrid...")
hybrid_cv_df, hybrid_history_df = run_cv_for_model(
    model_name="Hybrid",
    model_type="hybrid",
    build_fn=lambda: build_hybrid_patient_model(NUM_CLASSES, load_pretrained=True)
)

display(hybrid_cv_df)

In [ ]:
#13 5-fold 평균 +- 표준편차

all_cv_df = pd.concat([resnet_cv_df, vit_cv_df, hybrid_cv_df], ignore_index=True)

summary_df = (
    all_cv_df.groupby("model")[["ROC-AUC", "F1-score", "Recall"]]
    .agg(["mean", "std"])
)

display(summary_df)

In [ ]:
import matplotlib.pyplot as plt

def plot_one_model_by_fold(history_df, model_name):
    folds = sorted(history_df["fold"].unique())

    plt.figure(figsize=(12, 5))
    
    plt.subplot(1,2,1)
    for fold in folds:
        df_fold = history_df[history_df["fold"] == fold].sort_values("epoch")
        plt.plot(df_fold["epoch"], df_fold['train_loss'], linestyle='--', alpha = 0.7, label=f'Fold{fold} Train')
        plt.plot(df_fold["epoch"], df_fold['val_loss'], alpha = 0.7, label=f'Fold{fold} Val')
    plt.title(f"{model_name} - Loss by Fold")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend(fontsize=8)

    plt.subplot(1, 2, 2)
    for fold in folds:
        df_fold = history_df[history_df["fold"] == fold].sort_values("epoch")
        plt.plot(df_fold["epoch"], df_fold["val_auc"], alpha = 0.8, label = f"Fold{fold}")

    plt.title(f"{model_name} - Val ROC-AUC by Fold")
    plt.xlabel("Epoch")
    plt.ylabel("AUC")
    plt.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_one_model_by_fold(resnet_history_df, "ResNet50")
plot_one_model_by_fold(vit_history_df, "ViT")
plot_one_model_by_fold(hybrid_history_df, "Hybrid")